# Does each RDM's own geometry reproduce the tree?

**Question.** Notebooks `2_` to `5_` compare RDM *entries* with the a priori hierarchy. Here we ask about *structure*: if we let each RDM organise the images by itself (average-linkage clustering, no tree information), does the dendrogram it yields recover the Kiani-Mur tree?

**Why it matters.** Rank agreement can be high while the hierarchy is not recoverable (a gradient of similarity need not contain discrete nested categories), and the reverse. A hierarchy that a representation really *has* should show up when it is clustered blind. This also links s2 to s1, which clustered the behavioral embeddings at d = 3, 9, 15.

**Method in one paragraph.** Each complete RDM is clustered with average linkage. Whole-tree agreement is Baker's gamma against the exact tree (`path_tree_to_linkage` of the directory paths), with a null that shuffles image identity in the tree (1000 permutations, cached). Cophenetic correlation says how tree-like each RDM is. Flat cuts at each level's category count are scored with the adjusted Rand index (ARI) against the tree's categories.

**RDMs.** The complete members of the s2 set: `sem_km`, `sem_wn`, `clip_pre`, `clip_post`, `sens_pre`, `sens_post` and the two embeddings `behav_joint_mds_low` (d = 5) and `behav_joint_mds_high` (d = 20) (dimensionalities chosen in notebook 1 without the tree). The raw `behav_joint` has about 13% unobserved pairs and cannot be clustered, so its embedding stands for behavior. The s1 embeddings (d = 3, 9, 15) are added when their SMACOF fits are cached.

**Run from this directory.**

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from analysis.s0_build_rdms.common import RESULTS_DIR
from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import tree_recovery as tr

N_PERM = 1000
PARTS = c.tree_parts()

has_mds = all((RESULTS_DIR / f"D_{n}.npy").exists() for n in c.BEHAV_MDS_SET)
if not has_mds:
    print(f"{c.BEHAV_MDS_SET} do not exist yet: running on RAW_SET (behavior is not clustered until they do)")
rdms = c.load_rdm_set(c.FULL_SET if has_mds else c.RAW_SET)
complete, skipped = tr.split_complete(rdms)
print("not clustered (unobserved pairs):", skipped)

embeddings = tr.cached_embeddings(rdms[c.BEHAV])
if not embeddings:
    print(f"SMACOF fits at d = {tr.EMBEDDING_DIMS} are not cached yet: the s1 comparison is skipped")
clustered = {**complete, **embeddings}
trees = tr.rdm_trees(clustered)
print(f"{len(PARTS)} images; clustered: {list(clustered)}")

not clustered (unobserved pairs): ['behav_joint']


725 images; clustered: ['sem_km', 'sem_wn', 'clip_pre', 'clip_post', 'sens_pre', 'sens_post', 'behav_joint_mds_low', 'behav_joint_mds_high', 'behav_joint_mds3', 'behav_joint_mds9', 'behav_joint_mds15']


## 6.1 Does the dendrogram match the tree?

**Question.** How well does the merge order of each RDM's average-linkage dendrogram agree with the exact tree, in Baker's gamma (the correlation of the two dendrograms' cophenetic ranks; 1 = identical merge order, k-free)? **Why.** It is the one number that scores the whole hierarchy, not a single cut.

The grey bar is the 95% range of gamma when image identity is shuffled in the tree (keeping its shape): a dendrogram has to clear it to carry any tree-specific structure. `sem_km` is not exactly 1 because average linkage re-derives a tree from KM distances, which in a ragged tree are not an ultrametric.

In [2]:
gamma = tr.gamma_table(trees, PARTS, n_perm=N_PERM, cache_path=c.cache_path(f"tree_recovery_gamma_{len(trees)}rdms.npz"))
tr.fig_gamma(gamma).show()
print(gamma.round(3).to_string())

                      baker_gamma  null_mean  null_lo  null_hi  null_p95  p_value
rdm                                                                              
sem_km                      0.979       -0.0   -0.005    0.009     0.007    0.001
sem_wn                      0.306        0.0   -0.009    0.010     0.009    0.001
clip_pre                    0.329       -0.0   -0.010    0.013     0.010    0.001
clip_post                   0.277       -0.0   -0.011    0.013     0.010    0.001
sens_pre                    0.115        0.0   -0.010    0.013     0.011    0.001
sens_post                   0.049        0.0   -0.010    0.013     0.011    0.001
behav_joint_mds_low         0.680        0.0   -0.005    0.010     0.007    0.001
behav_joint_mds_high        0.804        0.0   -0.005    0.010     0.007    0.001
behav_joint_mds3            0.680        0.0   -0.005    0.010     0.007    0.001
behav_joint_mds9            0.805        0.0   -0.005    0.010     0.007    0.001
behav_joint_mds1

**Reading.** Every dendrogram clears its null by a wide margin (null 95% range about -0.01 to +0.01; smallest possible p = 0.001 with 1000 permutations), so all of them hold some tree-specific structure. The size differs greatly: the reference tree itself recovers 0.979. The behavioral embeddings come next (`behav_joint_mds_low`, d = 5: 0.680; `behav_joint_mds_high`, d = 20: 0.804; the s1 dimensionalities d = 3, 9, 15: 0.680, 0.805, 0.806, so gamma rises from low to high d and then plateaus). The semantic and CLIP RDMs sit near 0.3 (`sem_wn` 0.306, `clip_pre` 0.329, `clip_post` 0.277), and the pixel RDMs are close to the floor (`sens_pre` 0.115, `sens_post` 0.049). Significance says "not zero", not "recovers the tree": gamma is a rank correlation over 262,450 pairs, so a tiny value is already far outside the null.

## 6.2 How tree-like is each RDM?

**Question.** How much of an RDM's own geometry does its average-linkage dendrogram keep (Spearman correlation between the RDM and the cophenetic distances of its tree)? **Why.** A gamma computed from a dendrogram that barely summarises its RDM says little about that RDM. This is descriptive: it measures how well a hierarchy *fits* the geometry, not whether it is the *right* hierarchy.

In [3]:
fidelity = tr.cophenetic_table(trees, clustered)
print(pd.concat([fidelity, gamma["baker_gamma"]], axis=1).round(3).to_string())

                      cophenetic_fidelity  baker_gamma
sem_km                              0.933        0.979
sem_wn                              0.724        0.306
clip_pre                            0.663        0.329
clip_post                           0.667        0.277
sens_pre                            0.711        0.115
sens_post                           0.654        0.049
behav_joint_mds_low                 0.673        0.680
behav_joint_mds_high                0.773        0.804
behav_joint_mds3                    0.649        0.680
behav_joint_mds9                    0.722        0.805
behav_joint_mds15                   0.759        0.806


**Reading.** `sem_km` (0.933) is the most tree-like, as expected from construction; its remaining gap to 1 is the ragged-tree effect described above. The behavioral embeddings are 0.673 (`_low`, d = 5) and 0.773 (`_high`, d = 20), rising with d (d = 3: 0.649, d = 9: 0.722, d = 15: 0.759): a higher-dimensional embedding is a more tree-like summary. The other RDMs retain 0.65 to 0.72 (`sem_wn` 0.724, `sens_pre` 0.711, `clip_post` 0.667, `clip_pre` 0.663, `sens_post` 0.654), with no ordering that follows how semantic they are: the pixel RDMs are about as tree-like as CLIP. So their low gamma in 6.1 is not a failure to be tree-like; they are tree-like around a *different* hierarchy. For the embeddings, fidelity and gamma move together (d = 3 to 15), so a more tree-like embedding also matches the tree better.

## 6.3 Which level does a flat cut recover?

**Question.** Cut each dendrogram into as many clusters as the tree has categories at a level (2, 4 and 16 for domain, mid-level category and basic level) and score the partition against the tree's categories with ARI (0 = chance, 1 = identical). **Why.** Gamma pools every level; this shows where the match lives.

Because the plain cut is dominated by outlier clusters, an **outlier-robust cut** is reported alongside: the dendrogram is cut at the smallest k for which the number of clusters holding at least 5 images reaches the level's category count, and ARI is computed over the images in those clusters (this asks whether the substantial clusters are the tree's, and is descriptive; the share of images kept is printed).

Two caveats. Average linkage often peels off a few outliers rather than splitting the bulk, so the largest-cluster share (share of images in the biggest cluster) is printed: when it is near 1, the cut is nearly one cluster and ARI is near 0 for that reason. And the reference's own dendrogram, cut at 16, yields fewer clusters than asked (ties and raggedness), so it is a ceiling that is itself not 1.

In [4]:
cuts = tr.flat_cut_ari(trees, PARTS)
tr.fig_ari_heatmap(cuts).show()
for column, title in [("ari", "ARI"), ("largest_cluster", "share of images in the largest cluster"),
                      ("n_clusters", "clusters obtained"), ("robust_ari", "outlier-robust ARI"),
                      ("robust_k", "k of the outlier-robust cut"), ("robust_frac_kept", "share of images kept by the robust cut")]:
    print(f"\n{title} (columns: tree level)")
    print(cuts.pivot(index="rdm", columns="level", values=column).loc[list(clustered)].round(3).to_string())


ARI (columns: tree level)
level                     1      2      3
rdm                                      
sem_km                1.000  0.611  0.536
sem_wn               -0.022 -0.018  0.281
clip_pre             -0.009  0.120  0.286
clip_post            -0.002 -0.020  0.296
sens_pre             -0.004 -0.002  0.067
sens_post            -0.001 -0.006  0.099
behav_joint_mds_low   0.391  0.948  0.630
behav_joint_mds_high  0.391  0.956  0.625
behav_joint_mds3      0.391  0.956  0.644
behav_joint_mds9      0.391  0.964  0.615
behav_joint_mds15     0.391  0.968  0.648

share of images in the largest cluster (columns: tree level)
level                     1      2      3
rdm                                      
sem_km                0.589  0.589  0.327
sem_wn                0.924  0.923  0.354
clip_pre              0.983  0.840  0.487
clip_post             0.997  0.938  0.463
sens_pre              0.993  0.990  0.665
sens_post             0.999  0.959  0.553
behav_joint_mds_low   0.774  

**Reading.** *Plain cut.* At the domain cut the reference is 1.000 and at 4 clusters 0.611. Every non-behavioral RDM is at chance (ARI between -0.022 and -0.001) except `clip_pre` at the mid-level (0.120); their cuts are degenerate (largest cluster 92% to 100% of the images at the domain cut). The behavioral embeddings are the exception: domain ARI 0.391 for all of them (a 77% / 23% split against the tree's 59% / 41%), and mid-level ARI 0.948 (d = 5), 0.956 (d = 20), 0.956 / 0.964 / 0.968 (d = 3 / 9 / 15), with the largest cluster only 35% of the images. At 16 clusters the cuts are no longer degenerate: behavior 0.630 (d = 5) and 0.625 (d = 20), then `clip_post` 0.296, `clip_pre` 0.286, `sem_wn` 0.281, `sens_post` 0.099 and `sens_pre` 0.067 (the reference's own ceiling is 0.536, with 10 clusters).

*Outlier-robust cut.* It leaves the domain and the behavioral numbers essentially unchanged (behavior at 16 clusters 0.643 for both d = 5 and d = 20) and lifts the others where the plain cut was degenerate: mid-level `sem_wn` 0.277, `clip_pre` 0.101, `clip_post` 0.102, `sens_pre` 0.068, `sens_post` -0.007, and basic level `clip_pre` 0.558, `clip_post` 0.466, `sem_wn` 0.321, `sens_pre` 0.068, `sens_post` 0.102 (reference 0.697). It needs more clusters to find the substantial ones (k = 24 to 77 at the basic level for these RDMs, 19 to 26 for behavior) and keeps 88% to 100% of the images (the lowest are the pixel RDMs at the basic level, `sens_pre` 88% and `sens_post` 91%; the semantic and CLIP RDMs keep 93% to 97%, behavior 97% to 99%).

## Takeaways

* **Every dendrogram clears the image-shuffle null** (95% range about -0.01 to +0.01, p = 0.001 for all), but the effect size differs: Baker's gamma is 0.979 (reference), 0.804 (`behav_joint_mds_high`, d = 20), 0.680 (`behav_joint_mds_low`, d = 5), 0.329 (`clip_pre`), 0.306 (`sem_wn`), 0.277 (`clip_post`), 0.115 (`sens_pre`) and 0.049 (`sens_post`). Behavior recovers the tree's merge order far better than the semantic and CLIP RDMs, and the pixel RDMs barely.
* **s1 comparison.** The embeddings at d = 3, 9, 15 give gamma 0.680, 0.805, 0.806: a rise from 3 to 9, then a plateau, matching `_low` (d = 5) and `_high` (d = 20).
* **Tree-likeness only partly explains this.** Cophenetic correlations are 0.65 to 0.72 for the non-behavioral, non-reference RDMs (pixels as tree-like as CLIP): their geometry fits a hierarchy, just not this one. For behavior, fidelity rises with d (0.673 at d = 5, 0.773 at d = 20) together with gamma.
* **Flat cuts, plain.** Behavior recovers the mid-level almost perfectly (ARI 0.948 at d = 5, 0.956 at d = 20) and the basic level well (0.630, 0.625), but the domain cut only partly (0.391, an unbalanced 77% / 23% split). Others: basic level 0.296 (`clip_post`), 0.286 (`clip_pre`), 0.281 (`sem_wn`), 0.099 (`sens_post`), 0.067 (`sens_pre`); at 2 and 4 clusters they are about 0 (`clip_pre` at 4: 0.120) because average linkage isolates small groups.
* **Flat cuts, outlier-robust** (clusters of at least 5 images only): basic-level ARI is 0.643 for both behavioral embeddings, `clip_pre` 0.558 and `clip_post` 0.466 (reference 0.697), `sem_wn` 0.321, `sens_post` 0.102, `sens_pre` 0.068. Part of the CLIP coarse-cut failure is therefore outlier peeling, not the absence of the basic-level structure; the pixel RDMs stay near zero either way.
* **Limitations.** Gamma is dominated by the coarse structure (most pairs merge near the root), and the reference's own ceiling (gamma 0.979, plain basic-level ARI 0.536) is below 1 because KM distance on a ragged tree is not an ultrametric, so values are best read relative to `sem_km`. Behavior's advantage in the cuts is partly smoothing: an embedding denoises and completes the 13% unobserved pairs, which a raw RDM cannot. Flat cuts depend on the linkage and on the robust-cut rule (minimum cluster size 5). The image-shuffle null treats the tree as fixed and all numbers are point estimates (no bootstrap over images).